# Premier League Match Analysis: Home Advantage Study

## Research Question
**Does home advantage still exist in modern Premier League football, and what factors influence match outcomes?**

### Background
Home advantage is a well-documented phenomenon in sports. This analysis examines whether it persists in the Premier League and quantifies its magnitude.

### Hypotheses
1. **H1**: Home teams score significantly more goals than away teams
2. **H2**: Home teams have a win rate significantly greater than 50%
3. **H3**: Possession percentage correlates positively with match outcomes

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 1. Data Loading and Exploration

In [ ]:
# Load Premier League match data
df = pd.read_csv('../data/premier_league_matches_2023_24.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nTotal Matches: {len(df)}")
print(f"Gameweeks: {df['gameweek'].min()} to {df['gameweek'].max()}")
print(f"\nColumns: {list(df.columns)}")

df.head()

In [ ]:
# Basic statistics
print("Match Statistics Summary:")
df.describe()

## 2. H1: Home vs Away Goal Analysis

In [ ]:
# Compare home and away goals
home_goals_mean = df['home_goals'].mean()
away_goals_mean = df['away_goals'].mean()

print("H1: Home vs Away Goal Analysis")
print("="*50)
print(f"Home Goals: Mean = {home_goals_mean:.2f}, Std = {df['home_goals'].std():.2f}")
print(f"Away Goals: Mean = {away_goals_mean:.2f}, Std = {df['away_goals'].std():.2f}")
print(f"\nHome Goal Advantage: +{home_goals_mean - away_goals_mean:.2f} goals per match")

In [ ]:
# Paired t-test (home vs away in same matches)
t_stat, p_value = stats.ttest_rel(df['home_goals'], df['away_goals'], alternative='greater')

print(f"\nPaired t-test (Home > Away):")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("\n✓ H1 SUPPORTED: Home teams score significantly more goals")
else:
    print("\n✗ H1 NOT SUPPORTED: No significant difference in goals")

In [ ]:
# Effect size (Cohen's d for paired data)
diff = df['home_goals'] - df['away_goals']
cohens_d = diff.mean() / diff.std()

print(f"Effect Size (Cohen's d): {cohens_d:.3f}")

# Visualize goal distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution comparison
df['home_goals'].hist(bins=range(0, 8), alpha=0.6, label='Home Goals', ax=axes[0], color='steelblue', edgecolor='white')
df['away_goals'].hist(bins=range(0, 8), alpha=0.6, label='Away Goals', ax=axes[0], color='coral', edgecolor='white')
axes[0].set_xlabel('Goals Scored')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Home vs Away Goal Distribution')
axes[0].legend()

# Goal difference distribution
axes[1].hist(diff, bins=range(-6, 8), color='purple', alpha=0.7, edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].axvline(x=diff.mean(), color='green', linestyle='-', linewidth=2, label=f'Mean: {diff.mean():.2f}')
axes[1].set_xlabel('Goal Difference (Home - Away)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Goal Difference Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('../visualizations/epl_goals_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. H2: Home Win Rate Analysis

In [ ]:
# Calculate win rates
home_wins = (df['result'] == 'H').sum()
away_wins = (df['result'] == 'A').sum()
draws = (df['result'] == 'D').sum()
total_matches = len(df)

home_win_rate = home_wins / total_matches
away_win_rate = away_wins / total_matches
draw_rate = draws / total_matches

print("H2: Match Outcome Distribution")
print("="*50)
print(f"Total Matches: {total_matches}")
print(f"\nHome Wins: {home_wins} ({home_win_rate:.1%})")
print(f"Away Wins: {away_wins} ({away_win_rate:.1%})")
print(f"Draws: {draws} ({draw_rate:.1%})")

In [ ]:
# Binomial test: Is home win rate > 50%?
# Note: We exclude draws for a fair comparison
decisive_matches = home_wins + away_wins
binom_test = stats.binomtest(home_wins, decisive_matches, p=0.5, alternative='greater')

print(f"\nBinomial Test (H0: home win rate = 50%, excluding draws):")
print(f"Home wins in decisive matches: {home_wins}/{decisive_matches} ({home_wins/decisive_matches:.1%})")
print(f"p-value: {binom_test.pvalue:.6f}")

if binom_test.pvalue < 0.05:
    print("\n✓ H2 SUPPORTED: Home win rate is significantly > 50%")
else:
    print("\n✗ H2 NOT SUPPORTED: Home win rate not significantly > 50%")

In [ ]:
# Visualize outcomes
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart of outcomes
outcomes = [home_wins, draws, away_wins]
labels = ['Home Wins', 'Draws', 'Away Wins']
colors = ['#2ecc71', '#f39c12', '#e74c3c']

axes[0].pie(outcomes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90,
           explode=(0.05, 0, 0), shadow=True)
axes[0].set_title('Match Outcome Distribution', fontsize=14)

# Win rate by gameweek
weekly_home_wins = df.groupby('gameweek').apply(lambda x: (x['result'] == 'H').sum() / len(x))
axes[1].plot(weekly_home_wins.index, weekly_home_wins.values, 'o-', color='steelblue', linewidth=2)
axes[1].axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='50% threshold')
axes[1].axhline(y=home_win_rate, color='green', linestyle='-', linewidth=2, label=f'Overall: {home_win_rate:.1%}')
axes[1].set_xlabel('Gameweek')
axes[1].set_ylabel('Home Win Rate')
axes[1].set_title('Home Win Rate by Gameweek')
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../visualizations/epl_outcomes.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. H3: Possession and Match Outcomes

In [ ]:
# Create outcome variable (1 = home win, 0 = draw, -1 = away win)
df['home_outcome'] = df['result'].map({'H': 1, 'D': 0, 'A': -1})

# Correlation between possession and outcome
poss_outcome_corr, poss_p = stats.spearmanr(df['home_possession'], df['home_outcome'])

print("H3: Possession vs Match Outcome")
print("="*50)
print(f"Spearman Correlation: {poss_outcome_corr:.3f}")
print(f"p-value: {poss_p:.6f}")

if poss_outcome_corr > 0 and poss_p < 0.05:
    print("\n✓ H3 SUPPORTED: Higher possession correlates with better outcomes")
else:
    print("\n✗ H3 NOT SUPPORTED: No significant correlation")

In [ ]:
# Visualize possession vs outcome
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot of possession by outcome
outcome_map = {1: 'Home Win', 0: 'Draw', -1: 'Away Win'}
df['outcome_label'] = df['home_outcome'].map(outcome_map)

sns.boxplot(data=df, x='outcome_label', y='home_possession', 
            order=['Home Win', 'Draw', 'Away Win'],
            palette=['#2ecc71', '#f39c12', '#e74c3c'], ax=axes[0])
axes[0].set_xlabel('Match Outcome')
axes[0].set_ylabel('Home Possession (%)')
axes[0].set_title('Home Possession by Match Outcome')

# Shots vs goals
axes[1].scatter(df['home_shots'], df['home_goals'], alpha=0.5, label='Home', color='steelblue')
axes[1].scatter(df['away_shots'], df['away_goals'], alpha=0.5, label='Away', color='coral')

# Add regression lines
for data, color, label in [(df['home_shots'], 'steelblue', 'Home'), 
                           (df['away_shots'], 'coral', 'Away')]:
    z = np.polyfit(data, df['home_goals' if label == 'Home' else 'away_goals'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(data.min(), data.max(), 100)
    axes[1].plot(x_line, p(x_line), '--', color=color, linewidth=2)

axes[1].set_xlabel('Total Shots')
axes[1].set_ylabel('Goals Scored')
axes[1].set_title('Shots vs Goals Scored')
axes[1].legend()

plt.tight_layout()
plt.savefig('../visualizations/epl_possession_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Team Performance Analysis

In [ ]:
# Calculate team statistics
home_stats = df.groupby('home_team').agg({
    'home_goals': 'sum',
    'away_goals': 'sum',
    'home_points': 'sum',
    'home_shots': 'sum',
    'home_shots_on_target': 'sum'
}).rename(columns={'home_goals': 'goals_scored', 'away_goals': 'goals_conceded',
                    'home_points': 'points', 'home_shots': 'shots', 
                    'home_shots_on_target': 'shots_on_target'})

away_stats = df.groupby('away_team').agg({
    'away_goals': 'sum',
    'home_goals': 'sum',
    'away_points': 'sum',
    'away_shots': 'sum',
    'away_shots_on_target': 'sum'
}).rename(columns={'away_goals': 'goals_scored', 'home_goals': 'goals_conceded',
                    'away_points': 'points', 'away_shots': 'shots',
                    'away_shots_on_target': 'shots_on_target'})

# Combine home and away
team_stats = home_stats.add(away_stats)
team_stats['goal_diff'] = team_stats['goals_scored'] - team_stats['goals_conceded']
team_stats['conversion_rate'] = team_stats['goals_scored'] / team_stats['shots']
team_stats = team_stats.sort_values('points', ascending=False)

print("Team Performance Summary (Top 10 by Points):")
team_stats.head(10)

In [ ]:
# Visualize team performance
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# Points vs Goal Difference
top_teams = team_stats.head(10)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(top_teams)))

axes[0].barh(range(len(top_teams)), top_teams['points'], color=colors, edgecolor='white')
axes[0].set_yticks(range(len(top_teams)))
axes[0].set_yticklabels(top_teams.index)
axes[0].set_xlabel('Total Points')
axes[0].set_title('Top 10 Teams by Points')
axes[0].invert_yaxis()

# Goals scored vs conceded
axes[1].scatter(team_stats['goals_scored'], team_stats['goals_conceded'], 
                s=team_stats['points']*2, alpha=0.6, c=team_stats['goal_diff'],
                cmap='RdYlGn', edgecolors='white')

# Add team labels for top teams
for idx, row in team_stats.head(5).iterrows():
    axes[1].annotate(idx, (row['goals_scored'], row['goals_conceded']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)

axes[1].set_xlabel('Goals Scored')
axes[1].set_ylabel('Goals Conceded')
axes[1].set_title('Goals Scored vs Conceded (Size = Points)')

plt.tight_layout()
plt.savefig('../visualizations/epl_team_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Expected Goals (xG) Analysis

In [ ]:
# xG vs actual goals
df['home_xg_diff'] = df['home_goals'] - df['home_xg']
df['away_xg_diff'] = df['away_goals'] - df['away_xg']

print("Expected Goals (xG) Analysis")
print("="*50)
print(f"Home xG vs Actual: {df['home_xg'].mean():.2f} vs {df['home_goals'].mean():.2f}")
print(f"Away xG vs Actual: {df['away_xg'].mean():.2f} vs {df['away_goals'].mean():.2f}")
print(f"\nHome Finishing Efficiency: {df['home_xg_diff'].mean():.3f} goals above/below xG")
print(f"Away Finishing Efficiency: {df['away_xg_diff'].mean():.3f} goals above/below xG")

In [ ]:
# Visualize xG
fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(df['home_xg'], df['home_goals'], alpha=0.5, label='Home', color='steelblue')
ax.scatter(df['away_xg'], df['away_goals'], alpha=0.5, label='Away', color='coral')

# Perfect prediction line
max_val = max(df['home_xg'].max(), df['home_goals'].max(), df['away_xg'].max(), df['away_goals'].max())
ax.plot([0, max_val], [0, max_val], 'k--', linewidth=2, label='Perfect Prediction')

ax.set_xlabel('Expected Goals (xG)')
ax.set_ylabel('Actual Goals')
ax.set_title('xG vs Actual Goals')
ax.legend()

plt.tight_layout()
plt.savefig('../visualizations/epl_xg_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary and Conclusions

In [ ]:
print("="*70)
print("PREMIER LEAGUE HOME ADVANTAGE ANALYSIS - KEY FINDINGS")
print("="*70)

print("\n📊 DATASET OVERVIEW:")
print(f"   • {len(df)} matches analyzed")
print(f"   • {df['home_team'].nunique()} teams")
print(f"   • Gameweeks {df['gameweek'].min()}-{df['gameweek'].max()}")

print("\n🔬 HYPOTHESIS TESTING RESULTS:")

print(f"\n   H1: Home vs Away Goals")
print(f"       • Home goals: {home_goals_mean:.2f}/match")
print(f"       • Away goals: {away_goals_mean:.2f}/match")
print(f"       • Home advantage: +{home_goals_mean - away_goals_mean:.2f} goals/match")
print(f"       • Result: {'SUPPORTED' if p_value < 0.05 else 'NOT SUPPORTED'}")

print(f"\n   H2: Home Win Rate")
print(f"       • Home win rate: {home_win_rate:.1%}")
print(f"       • Result: {'SUPPORTED' if binom_test.pvalue < 0.05 else 'NOT SUPPORTED'}")

print(f"\n   H3: Possession-Outcome Correlation")
print(f"       • Spearman r: {poss_outcome_corr:.3f}")
print(f"       • Result: {'SUPPORTED' if poss_outcome_corr > 0 and poss_p < 0.05 else 'NOT SUPPORTED'}")

print("\n📈 KEY INSIGHTS:")
print(f"   • Home advantage yields ~{(home_goals_mean - away_goals_mean):.2f} extra goals per match")
print(f"   • Home teams win {home_win_rate:.0%} of matches")
print(f"   • Possession has {'weak' if abs(poss_outcome_corr) < 0.3 else 'moderate'} correlation with outcomes")

print("\n" + "="*70)

## Conclusions

This analysis examined home advantage in the Premier League:

1. **Home Goal Advantage**: Home teams score more goals on average, confirming the existence of home advantage.

2. **Win Rate**: Home teams win significantly more than 50% of decisive matches.

3. **Possession**: While possession correlates with outcomes, the relationship is not deterministic - efficient teams can win with less possession.

**Practical Implications**:
- Teams should factor home advantage into tactical planning
- xG provides a useful metric for evaluating finishing quality
- Traditional possession stats don't guarantee results

**Limitations**:
- Synthetic data for demonstration
- Single season analysis
- No account for team strength differences

**Future Analysis**:
- Multi-season trends
- Impact of crowd size (post-COVID analysis)
- Tactical formations and outcomes